# Counter-Strike Enemy Detection - YOLOv11m Fine-tuning
## Optimized for Google Colab Free Tier (T4 GPU)

This notebook fine-tunes YOLOv11m on OptShot dataset for CS enemy detection.

## 1. Setup Environment & Check GPU

In [ ]:
# Check GPU availability and specs
!nvidia-smi

In [ ]:
# Install required packages (optimized versions for Colab)
!pip install -q ultralytics==8.3.0  # Latest stable version with YOLOv11
!pip install -q roboflow  # In case dataset needs conversion

# Import libraries
import os
import torch
import yaml
from ultralytics import YOLO
from IPython.display import Image, display

# Verify CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 2. Download and Prepare Dataset

In [ ]:
# Clone OptShot dataset
!git clone https://github.com/OptShot/OptShot.git

# Navigate to dataset directory
%cd OptShot

In [ ]:
# Explore dataset structure
!ls -la

In [ ]:
# Check if data.yaml exists or needs to be created
# This cell will help understand the dataset structure
import os
from pathlib import Path

# Find all directories
for root, dirs, files in os.walk('.'):
    level = root.replace('.', '', 1).count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Show first 5 files only
        print(f'{subindent}{file}')
    if len(files) > 5:
        print(f'{subindent}... and {len(files) - 5} more files')
    if level > 2:  # Limit depth
        break

In [ ]:
# Create or verify data.yaml configuration
# Adjust paths based on actual dataset structure

data_yaml_content = """
# Dataset paths (adjust if needed)
path: /content/OptShot  # Root directory
train: images/train  # Training images relative to 'path'
val: images/val      # Validation images relative to 'path'

# Classes
names:
  0: enemy  # Enemy player (CT or T)

# Number of classes
nc: 1
"""

# Check if data.yaml exists
if not os.path.exists('data.yaml'):
    print("Creating data.yaml...")
    with open('data.yaml', 'w') as f:
        f.write(data_yaml_content)
    print("data.yaml created!")
else:
    print("data.yaml already exists")
    with open('data.yaml', 'r') as f:
        print(f.read())

## 3. Initialize YOLOv11m Model

In [ ]:
# Load YOLOv11m pretrained model
# The model will be downloaded automatically on first run
model = YOLO('yolo11m.pt')  # Medium model for balance between speed and accuracy

print("Model loaded successfully!")
print(f"Model summary:")
model.info()  # Display model information

## 4. Training Configuration (Optimized for T4 GPU)

In [ ]:
# Training parameters optimized for Colab T4 (16GB VRAM)
# These settings balance training speed, memory usage, and model performance

results = model.train(
    data='data.yaml',              # Path to data config
    epochs=100,                     # Number of training epochs (adjust based on time)
    imgsz=640,                      # Input image size (640 is optimal for speed/accuracy)
    batch=16,                       # Batch size (T4 can handle 16 for yolo11m at 640)
    patience=20,                    # Early stopping patience
    save=True,                      # Save checkpoints
    device=0,                       # Use GPU 0
    workers=2,                      # Data loader workers (Colab has limited CPU)
    project='runs/cs_detection',    # Project directory
    name='yolo11m_optshot',         # Run name
    exist_ok=True,                  # Overwrite existing project
    pretrained=True,                # Use pretrained weights
    optimizer='AdamW',              # Optimizer (AdamW is robust)
    lr0=0.001,                      # Initial learning rate
    lrf=0.01,                       # Final learning rate (lr0 * lrf)
    momentum=0.937,                 # SGD momentum/Adam beta1
    weight_decay=0.0005,            # Weight decay
    warmup_epochs=3.0,              # Warmup epochs
    warmup_momentum=0.8,            # Warmup momentum
    box=7.5,                        # Box loss gain
    cls=0.5,                        # Class loss gain
    dfl=1.5,                        # DFL loss gain
    hsv_h=0.015,                    # HSV-Hue augmentation
    hsv_s=0.7,                      # HSV-Saturation augmentation
    hsv_v=0.4,                      # HSV-Value augmentation
    degrees=0.0,                    # Rotation augmentation (disabled for FPS games)
    translate=0.1,                  # Translation augmentation
    scale=0.5,                      # Scale augmentation
    shear=0.0,                      # Shear augmentation (disabled)
    perspective=0.0,                # Perspective augmentation (disabled)
    flipud=0.0,                     # Flip up-down (disabled for FPS)
    fliplr=0.5,                     # Flip left-right probability
    mosaic=1.0,                     # Mosaic augmentation probability
    mixup=0.1,                      # Mixup augmentation probability
    copy_paste=0.0,                 # Copy-paste augmentation
    cache=False,                    # Cache images (False to save RAM on Colab)
    amp=True,                       # Automatic Mixed Precision (faster training)
    fraction=1.0,                   # Dataset fraction to use
    save_period=10,                 # Save checkpoint every N epochs
    val=True,                       # Validate during training
    plots=True,                     # Generate training plots
    verbose=True                    # Verbose output
)

print("\nTraining completed!")

## 5. Evaluate Results

In [ ]:
# Display training results
print("Training Metrics:")
print(f"Best mAP50: {results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"Best mAP50-95: {results.results_dict['metrics/mAP50-95(B)']:.4f}")

# Display training plots
from IPython.display import Image, display

print("\n=== Training Results ===")
display(Image(filename='runs/cs_detection/yolo11m_optshot/results.png'))

print("\n=== Confusion Matrix ===")
display(Image(filename='runs/cs_detection/yolo11m_optshot/confusion_matrix.png'))

print("\n=== Validation Batch Predictions ===")
display(Image(filename='runs/cs_detection/yolo11m_optshot/val_batch0_pred.jpg'))

## 6. Validate on Test Set

In [ ]:
# Load best trained model
best_model = YOLO('runs/cs_detection/yolo11m_optshot/weights/best.pt')

# Run validation
metrics = best_model.val(data='data.yaml')

# Print metrics
print(f"\nValidation Metrics:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

## 7. Test Inference on Sample Images

In [ ]:
# Test inference on validation images
import glob

# Get sample validation images
val_images = glob.glob('images/val/*.jpg')[:5] + glob.glob('images/val/*.png')[:5]

if val_images:
    # Run inference
    results = best_model.predict(
        source=val_images,
        conf=0.25,                  # Confidence threshold
        iou=0.45,                   # NMS IoU threshold
        imgsz=640,                  # Image size
        save=True,                  # Save results
        project='runs/cs_detection',
        name='test_predictions'
    )
    
    print(f"\nInference completed on {len(val_images)} images")
    print(f"Results saved to: runs/cs_detection/test_predictions/")
    
    # Display first prediction
    pred_images = glob.glob('runs/cs_detection/test_predictions/*.jpg')
    if pred_images:
        display(Image(filename=pred_images[0]))
else:
    print("No validation images found. Check dataset structure.")

## 8. Export Model for Deployment

In [ ]:
# Export to different formats for deployment

# Export to ONNX (for cross-platform inference)
print("Exporting to ONNX...")
best_model.export(format='onnx', imgsz=640, simplify=True)

# Export to TorchScript (for PyTorch deployment)
print("\nExporting to TorchScript...")
best_model.export(format='torchscript', imgsz=640)

# Export to TensorRT (for NVIDIA GPU inference - optional)
# Uncomment if you need TensorRT
# print("\nExporting to TensorRT...")
# best_model.export(format='engine', imgsz=640, half=True)

print("\nExports completed!")
print(f"Models saved in: runs/cs_detection/yolo11m_optshot/weights/")

## 9. Download Trained Models (Optional)

In [ ]:
# Zip weights for download
!zip -r cs_yolo11m_weights.zip runs/cs_detection/yolo11m_optshot/weights/

print("\nWeights zipped! Download 'cs_yolo11m_weights.zip' from Colab files panel.")

# Or use Colab's download function
from google.colab import files
files.download('cs_yolo11m_weights.zip')

## 10. Quick Inference Demo

In [ ]:
# Quick demo: Load model and run inference
from ultralytics import YOLO

# Load trained model
demo_model = YOLO('runs/cs_detection/yolo11m_optshot/weights/best.pt')

# Single image inference example
# Replace 'path/to/image.jpg' with actual test image path
# results = demo_model.predict('path/to/image.jpg', conf=0.25, save=True)

print("Model ready for inference!")
print(f"\nUsage example:")
print("results = demo_model.predict('image.jpg', conf=0.25)")
print("for r in results:")
print("    boxes = r.boxes  # Bounding boxes")
print("    print(f'Detected {len(boxes)} enemies')")